# LeetCode 1848 — Minimum Distance to the Target Element bằng Python

Link đề: https://leetcode.com/problems/minimum-distance-to-the-target-element/

Độ khó: **Easy** · Chủ đề: Array

Skeleton chữ ký LeetCode:

```python
class Solution:
    def getMinDistance(self, nums: List[int], target: int, start: int) -> int:

```

**Mục tiêu học**

1. Phân biệt rõ **giá trị** `nums[i]` và **chỉ số** `i`; hiểu khoảng cách `abs(i - start)`.
2. Ôn `enumerate`, `abs`, `min` với generator, `float('inf')`.
3. Học 4 cách: quét toàn bộ lấy min → `min(...)` một dòng → gom chỉ số rồi chọn → **lan ra hai phía từ `start`** (dừng sớm).
4. Chứng minh vì sao cách lan hai phía trả đúng ngay khi gặp `target` lần đầu.
5. Trace, Big-O (worst vs best), lỗi hay gặp và 6 bài tập về "tìm phần tử gần nhất".

## 1. Bài toán yêu cầu gì?

Cho mảng `nums` (chỉ số từ 0), số `target` và chỉ số `start`. Tìm chỉ số `i` sao cho `nums[i] == target` và `abs(i - start)` **nhỏ nhất**. Trả về **giá trị** `abs(i - start)` (không phải `i`).

Đề **đảm bảo** `target` có trong `nums`.

| nums | target | start | Kết quả | Vì sao |
|---|---:|---:|---:|---|
| `[1,2,3,4,5]` | 5 | 3 | 1 | chỉ có `nums[4]=5`, `|4-3|=1` |
| `[1]` | 1 | 0 | 0 | `|0-0|=0` |
| `[1,1,1,1,1,1,1,1,1,1]` | 1 | 0 | 0 | `nums[0]` đã là 1 |

**Constraints:** `1 <= len(nums) <= 1000`, `1 <= nums[i] <= 10^4`, `0 <= start < len(nums)`, `target` có trong `nums`.

In [1]:
from typing import List

examples = [
    (([1, 2, 3, 4, 5], 5, 3), 1),
    (([1], 1, 0), 0),
    (([1] * 10, 1, 0), 0),
]
for (nums, target, start), expected in examples:
    print(f"nums={nums}, target={target}, start={start} -> {expected}")

nums=[1, 2, 3, 4, 5], target=5, start=3 -> 1
nums=[1], target=1, start=0 -> 0
nums=[1, 1, 1, 1, 1, 1, 1, 1, 1, 1], target=1, start=0 -> 0


## 2. Nền tảng Python

### 2.1. Chỉ số khác giá trị — `enumerate`

`enumerate(nums)` cho cặp `(i, nums[i])`. Bài này hỏi về **chỉ số** gần `start`, nên ta cần cả `i` lẫn giá trị.

In [2]:
nums = [5, 3, 6, 5]
for i, x in enumerate(nums):
    print(f"chỉ số i={i}, giá trị nums[i]={x}")

chỉ số i=0, giá trị nums[i]=5
chỉ số i=1, giá trị nums[i]=3
chỉ số i=2, giá trị nums[i]=6
chỉ số i=3, giá trị nums[i]=5


### 2.2. `abs()` — khoảng cách không âm

Khoảng cách giữa hai chỉ số không quan tâm bên trái hay phải: `abs(2 - 5) = abs(5 - 2) = 3`.

In [3]:
print(abs(2 - 5), abs(5 - 2), abs(0))
start = 3
for i in range(6):
    print(f"i={i}: |i - start| = {abs(i - start)}")

3 3 0
i=0: |i - start| = 3
i=1: |i - start| = 2
i=2: |i - start| = 1
i=3: |i - start| = 0
i=4: |i - start| = 1
i=5: |i - start| = 2


### 2.3. `min` với generator và `float('inf')`

- `min(biểu_thức for ... if ...)` tìm min của các giá trị thỏa điều kiện, không cần tạo list.
- Nếu tự cập nhật min trong vòng lặp, khởi tạo `best = float('inf')` (dương vô cùng) để giá trị đầu tiên chắc chắn nhỏ hơn.

In [4]:
vals = [7, 2, 9]
print(min(v for v in vals if v > 3))
best = float("inf")
for v in vals:
    best = min(best, v)
print("best =", best, "| inf > 10**18:", float("inf") > 10**18)

7
best = 2 | inf > 10**18: True


### 2.4. Kiểm tra chỉ số hợp lệ

Khi đi ra hai phía từ `start`, ta sẽ xét `start - d` và `start + d`. Cần kiểm `0 <= j < len(nums)` trước khi đọc `nums[j]`. Đặc biệt **chỉ số âm** không báo lỗi trong Python mà âm thầm đọc từ cuối mảng!

In [5]:
nums = [10, 20, 30]
print("nums[-1] =", nums[-1], "(không lỗi, nhưng là phần tử cuối!)")
try:
    nums[3]
except IndexError as e:
    print("nums[3] ->", e)
j = -1
print("0 <= j < len(nums)?", 0 <= j < len(nums))

nums[-1] = 30 (không lỗi, nhưng là phần tử cuối!)
nums[3] -> list index out of range
0 <= j < len(nums)? False


Python cho phép **so sánh nối** `0 <= j < n`, nghĩa là `0 <= j and j < n`.

## 3. Cách 1 — quét toàn bộ, giữ min (baseline)

Ý tưởng tự nhiên: xét **mọi** `i`; nếu `nums[i] == target` thì tính `abs(i - start)` và cập nhật min.

In [6]:
def min_distance_scan(nums: List[int], target: int, start: int) -> int:
    best = float("inf")
    for i, x in enumerate(nums):
        if x == target:
            best = min(best, abs(i - start))
    return best

for (nums, target, start), expected in examples:
    print(min_distance_scan(nums, target, start), "| expected", expected)
    assert min_distance_scan(nums, target, start) == expected

1 | expected 1
0 | expected 0
0 | expected 0


### 3.1. Viết gọn bằng `min(...)`

Vì đề đảm bảo có `target`, generator không bao giờ rỗng, `min` không lỗi.

In [7]:
def min_distance_oneliner(nums, target, start):
    return min(abs(i - start) for i, x in enumerate(nums) if x == target)

print(min_distance_oneliner([5, 3, 6], 5, 2))
try:
    min(abs(i) for i, x in enumerate([1, 2]) if x == 99)
except ValueError as e:
    print("Nếu target không có:", e)

2
Nếu target không có: min() iterable argument is empty


### 3.2. Cách 2 — gom các chỉ số rồi chọn

Tách bài thành hai bước dễ debug: (1) lấy list chỉ số của `target`; (2) chọn chỉ số gần `start` nhất. `min(list, key=hàm)` chọn phần tử có `hàm(phần tử)` nhỏ nhất.

In [8]:
def min_distance_collect(nums, target, start):
    positions = [i for i, x in enumerate(nums) if x == target]
    closest = min(positions, key=lambda i: abs(i - start))
    return abs(closest - start)

nums, target, start = [5, 1, 5, 2, 5], 5, 3
print("positions =", [i for i, x in enumerate(nums) if x == target])
print("đáp án =", min_distance_collect(nums, target, start))

positions = [0, 2, 4]
đáp án = 1


`lambda i: abs(i - start)` là **hàm vô danh** viết một dòng, tương đương `def f(i): return abs(i - start)`.

### 3.3. Nút thắt

Cả ba cách trên luôn đọc **hết** mảng (`n` phần tử), kể cả khi `target` nằm ngay cạnh `start`. Ví dụ `nums = [1]*1000`, `start = 500`: đáp án 0 có ngay ở bước đầu, nhưng vẫn quét 1000 lần.

Câu hỏi: có thể **xét các chỉ số theo thứ tự khoảng cách tăng dần** để dừng ở lần gặp đầu tiên không?

In [9]:
def count_reads_scan(nums, target, start):
    reads = 0
    for i, x in enumerate(nums):
        reads += 1
    return reads

print("đọc:", count_reads_scan([1] * 1000, 1, 500), "phần tử dù đáp án là 0")

đọc: 1000 phần tử dù đáp án là 0


## 4. Insight — lan ra hai phía (như gợi ý đề)

Xét `d = 0, 1, 2, ...`. Với mỗi `d`, chỉ có tối đa **hai** chỉ số cách `start` đúng `d`: `start - d` và `start + d`.

Nếu duyệt `d` tăng dần thì **lần đầu** gặp `target` chính là khoảng cách **nhỏ nhất** — trả về `d` ngay.

Hình dung: thả viên đá xuống nước tại `start`, gợn sóng lan đều ra hai bên; sóng chạm `target` đầu tiên ở đâu thì đó là gần nhất.

In [10]:
start, n = 3, 7
for d in range(0, 4):
    left, right = start - d, start + d
    print(f"d={d}: xét chỉ số {left} và {right}")

d=0: xét chỉ số 3 và 3
d=1: xét chỉ số 2 và 4
d=2: xét chỉ số 1 và 5
d=3: xét chỉ số 0 và 6


## 5. Lời giải chính — lan hai phía, giảng từng đoạn

**Đoạn 1 — vòng khoảng cách:** `for d in range(len(nums))`. Khoảng cách lớn nhất có thể là `n - 1`.

**Đoạn 2 — phía trái:** `j = start - d`; nếu `j >= 0` và `nums[j] == target` ⇒ trả `d`.

**Đoạn 3 — phía phải:** `j = start + d`; nếu `j < n` và `nums[j] == target` ⇒ trả `d`.

Với `d = 0`, hai phía trùng nhau ở `start` — kiểm hai lần cũng không sao.

In [11]:
def min_distance_expand(nums: List[int], target: int, start: int) -> int:
    n = len(nums)
    for d in range(n):
        left = start - d
        if left >= 0 and nums[left] == target:
            return d
        right = start + d
        if right < n and nums[right] == target:
            return d
    return -1   # không xảy ra vì đề đảm bảo target có trong nums

for (nums, target, start), expected in examples:
    assert min_distance_expand(nums, target, start) == expected
print(min_distance_expand([5, 3, 6], 5, 2), min_distance_expand([1, 2, 3, 4, 5], 5, 3))

2 1


Thứ tự `left >= 0 and nums[left] == target` quan trọng: `and` **ngắn mạch** (short-circuit) — nếu vế trái `False` thì không đánh giá vế phải, nên không đọc `nums[-1]` nhầm.

In [12]:
nums = [7, 8]
left = -1
print(left >= 0 and nums[left] == 8)     # False, không đọc nums[-1]
print(nums[left] == 8)                    # True!? vì nums[-1] = 8 — nếu quên kiểm biên

False
True


## 6. TRACE từng bước

Trace `nums = [4, 9, 2, 9, 7, 1, 4]`, `target = 4`, `start = 3`. Các chỉ số có 4 là 0 và 6, cả hai cách `start` 3 đơn vị.

| d | left | nums[left] | right | nums[right] | kết luận |
|---:|---:|---:|---:|---:|---|
| 0 | 3 | 9 | 3 | 9 | chưa thấy |
| 1 | 2 | 2 | 4 | 7 | chưa thấy |
| 2 | 1 | 9 | 5 | 1 | chưa thấy |
| 3 | 0 | **4** | — | — | trả `3` (không cần xét phải) |

In [13]:
def trace_expand(nums, target, start):
    n = len(nums)
    print(f"nums={nums}, target={target}, start={start}")
    for d in range(n):
        left, right = start - d, start + d
        lv = nums[left] if left >= 0 else "ngoài"
        print(f"d={d}: left={left:>2} ({lv})", end="")
        if left >= 0 and nums[left] == target:
            print("  -> gặp target bên trái, trả", d)
            return d
        rv = nums[right] if right < n else "ngoài"
        print(f" | right={right:>2} ({rv})", end="")
        if right < n and nums[right] == target:
            print("  -> gặp target bên phải, trả", d)
            return d
        print("  -> chưa thấy")

assert trace_expand([4, 9, 2, 9, 7, 1, 4], 4, 3) == 3

nums=[4, 9, 2, 9, 7, 1, 4], target=4, start=3
d=0: left= 3 (9) | right= 3 (9)  -> chưa thấy
d=1: left= 2 (2) | right= 4 (7)  -> chưa thấy
d=2: left= 1 (9) | right= 5 (1)  -> chưa thấy
d=3: left= 0 (4)  -> gặp target bên trái, trả 3


Trace thứ hai: `start` sát biên trái, phía trái nhanh chóng ra ngoài mảng — thấy nhánh "ngoài" hoạt động.

In [14]:
assert trace_expand([3, 1, 1, 1, 5], 5, 0) == 4
print()
assert trace_expand([5, 3, 6], 5, 2) == 2

nums=[3, 1, 1, 1, 5], target=5, start=0
d=0: left= 0 (3) | right= 0 (3)  -> chưa thấy
d=1: left=-1 (ngoài) | right= 1 (1)  -> chưa thấy
d=2: left=-2 (ngoài) | right= 2 (1)  -> chưa thấy
d=3: left=-3 (ngoài) | right= 3 (1)  -> chưa thấy
d=4: left=-4 (ngoài) | right= 4 (5)  -> gặp target bên phải, trả 4

nums=[5, 3, 6], target=5, start=2
d=0: left= 2 (6) | right= 2 (6)  -> chưa thấy
d=1: left= 1 (3) | right= 3 (ngoài)  -> chưa thấy
d=2: left= 0 (5)  -> gặp target bên trái, trả 2


## 7. Cách 4 — lan hai phía với `while` và hai con trỏ

Một cách viết khác: hai con trỏ `l` và `r` bắt đầu ở `start`, mỗi bước `l -= 1`, `r += 1`. Dừng khi cả hai đều ra ngoài.

In [15]:
def min_distance_two_pointers(nums, target, start):
    n = len(nums)
    l = r = start
    while l >= 0 or r < n:
        if l >= 0 and nums[l] == target:
            return start - l
        if r < n and nums[r] == target:
            return r - start
        l -= 1
        r += 1
    return -1

print(min_distance_two_pointers([4, 9, 2, 9, 7, 1, 4], 4, 3))

3


`l = r = start` gán **cùng một giá trị** cho hai biến (với số nguyên không có vấn đề "cùng tham chiếu" vì số bất biến).

## 8. Code nộp LeetCode

### Cách A — quét toàn bộ (đơn giản, luôn đúng)

In [16]:
class Solution:
    def getMinDistance(self, nums: List[int], target: int, start: int) -> int:
        return min(abs(i - start) for i, x in enumerate(nums) if x == target)

print(Solution().getMinDistance([1, 2, 3, 4, 5], 5, 3))

1


### Cách B — lan hai phía (dừng sớm)

In [17]:
class SolutionExpand:
    def getMinDistance(self, nums: List[int], target: int, start: int) -> int:
        n = len(nums)
        for d in range(n):
            if start - d >= 0 and nums[start - d] == target:
                return d
            if start + d < n and nums[start + d] == target:
                return d
        return -1

print(SolutionExpand().getMinDistance([1, 2, 3, 4, 5], 5, 3))

1


### Cách C — quét có dừng sớm khi đạt 0

In [18]:
class SolutionScanEarly:
    def getMinDistance(self, nums: List[int], target: int, start: int) -> int:
        best = len(nums)
        for i, x in enumerate(nums):
            if x == target:
                best = min(best, abs(i - start))
                if best == 0:
                    break
        return best

print(SolutionScanEarly().getMinDistance([1] * 10, 1, 0))

0


`break` thoát vòng lặp ngay. Khởi tạo `best = len(nums)` cũng an toàn vì mọi khoảng cách hợp lệ `<= n - 1`.

## 9. Vì sao đúng?

**Cách quét:** xét **mọi** chỉ số có `target`, lấy min — đúng theo định nghĩa.

**Cách lan hai phía:** Invariant — khi bắt đầu vòng `d`, **không** có chỉ số `j` nào với `|j - start| < d` mà `nums[j] == target` (vì mọi chỉ số đó đã được xét ở các vòng `0..d-1`, mỗi vòng xét đúng hai chỉ số `start ± d'`).

Khi gặp `target` ở vòng `d`, có một chỉ số với khoảng cách `d`, và không có chỉ số nào gần hơn ⇒ `d` là min.

**Dừng chắc chắn:** `target` có trong mảng ở chỉ số `j` nào đó, và `|j - start| <= n - 1`, nên vòng `d = |j - start|` sẽ gặp nó (nếu chưa gặp sớm hơn).

In [19]:
# Kiểm chứng invariant: mọi chỉ số được xét đúng theo thứ tự khoảng cách không giảm
def visit_order(n, start):
    order = []
    for d in range(n):
        for j in (start - d, start + d):
            if 0 <= j < n and j not in order:
                order.append(j)
    return order

order = visit_order(7, 2)
print("thứ tự xét:", order)
print("khoảng cách:", [abs(j - 2) for j in order])
assert sorted(order) == list(range(7))

thứ tự xét: [2, 1, 3, 0, 4, 5, 6]
khoảng cách: [0, 1, 1, 2, 2, 3, 4]


## 10. Độ phức tạp

Gọi `n = len(nums)`, `D` = đáp án.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| Quét + min | `O(n)` luôn | `O(1)` | đọc toàn bộ |
| Gom chỉ số | `O(n)` | `O(k)` (k = số lần xuất hiện) | dễ debug |
| Lan hai phía | `O(D)` ≤ `O(n)` | `O(1)` | nhanh khi target gần start |
| Hai con trỏ | `O(D)` | `O(1)` | như trên |

Worst case của cách lan hai phía vẫn là `O(n)` (target ở đầu xa nhất). Nhưng best case là `O(1)`.

In [20]:
def count_reads_expand(nums, target, start):
    n, reads = len(nums), 0
    for d in range(n):
        candidates = (start,) if d == 0 else (start - d, start + d)   # d = 0: chỉ một chỉ số
        for j in candidates:
            if 0 <= j < n:
                reads += 1
                if nums[j] == target:
                    return reads
    return reads

big = [1] * 1000
print("lan hai phía đọc:", count_reads_expand(big, 1, 500), "phần tử")
worst = [2] * 999 + [1]
print("worst case đọc:", count_reads_expand(worst, 1, 0), "phần tử")

lan hai phía đọc: 1 phần tử
worst case đọc: 1000 phần tử


## 11. Bộ test

Nhóm: mẫu đề; target ngay tại start; target ở hai phía cách đều; nhiều lần xuất hiện; start ở biên trái/phải; mảng một phần tử; ngẫu nhiên đối chiếu.

In [21]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<24} PASS ({actual})")

for Impl in [Solution, SolutionExpand, SolutionScanEarly]:
    s = Impl()
    print("==", Impl.__name__)
    check("mẫu 1", s.getMinDistance([1, 2, 3, 4, 5], 5, 3), 1)
    check("mẫu 2", s.getMinDistance([1], 1, 0), 0)
    check("mẫu 3", s.getMinDistance([1] * 10, 1, 0), 0)
    check("mẫu testcase [5,3,6]", s.getMinDistance([5, 3, 6], 5, 2), 2)
    check("cách đều hai phía", s.getMinDistance([4, 9, 2, 9, 7, 1, 4], 4, 3), 3)
    check("start ở biên phải", s.getMinDistance([7, 1, 1, 1], 7, 3), 3)
    check("gần bên phải hơn", s.getMinDistance([2, 0, 0, 0, 2, 0], 2, 3), 1)

== Solution
mẫu 1                    PASS (1)
mẫu 2                    PASS (0)
mẫu 3                    PASS (0)
mẫu testcase [5,3,6]     PASS (2)
cách đều hai phía        PASS (3)
start ở biên phải        PASS (3)
gần bên phải hơn         PASS (1)
== SolutionExpand
mẫu 1                    PASS (1)
mẫu 2                    PASS (0)
mẫu 3                    PASS (0)
mẫu testcase [5,3,6]     PASS (2)
cách đều hai phía        PASS (3)
start ở biên phải        PASS (3)
gần bên phải hơn         PASS (1)
== SolutionScanEarly
mẫu 1                    PASS (1)
mẫu 2                    PASS (0)
mẫu 3                    PASS (0)
mẫu testcase [5,3,6]     PASS (2)
cách đều hai phía        PASS (3)
start ở biên phải        PASS (3)
gần bên phải hơn         PASS (1)


In [22]:
import random
rng = random.Random(1848)
impls = [min_distance_scan, min_distance_oneliner, min_distance_collect, min_distance_expand,
         min_distance_two_pointers, Solution().getMinDistance, SolutionExpand().getMinDistance,
         SolutionScanEarly().getMinDistance]
for _ in range(2000):
    n = rng.randint(1, 30)
    nums = [rng.randint(1, 4) for _ in range(n)]
    target = rng.choice(nums)
    start = rng.randrange(n)
    res = {f(nums, target, start) for f in impls}
    assert len(res) == 1, (nums, target, start, res)
print("8 cài đặt đồng ý trên 2000 bộ ngẫu nhiên PASS")

8 cài đặt đồng ý trên 2000 bộ ngẫu nhiên PASS


## 12. Lỗi người mới hay gặp

### Lỗi 1: trả về chỉ số `i` thay vì khoảng cách

In [23]:
def wrong_return_index(nums, target, start):
    best_i, best = None, float("inf")
    for i, x in enumerate(nums):
        if x == target and abs(i - start) < best:
            best, best_i = abs(i - start), i
    return best_i

print("sai :", wrong_return_index([1, 2, 3, 4, 5], 5, 3), "| đúng:", min_distance_scan([1, 2, 3, 4, 5], 5, 3))

sai : 4 | đúng: 1


Đề yêu cầu `abs(i - start)`. Sửa: trả `best`.

### Lỗi 2: quên `abs`

In [24]:
def wrong_no_abs(nums, target, start):
    return min(i - start for i, x in enumerate(nums) if x == target)

print("sai :", wrong_no_abs([5, 0, 0, 5], 5, 2), "| đúng:", min_distance_scan([5, 0, 0, 5], 5, 2))

sai : -2 | đúng: 1


Chỉ số bên trái cho hiệu **âm**, `min` chọn nó dù xa hơn. Sửa: `abs(i - start)`.

### Lỗi 3: lan hai phía nhưng không kiểm biên trái

In [25]:
def wrong_no_left_bound(nums, target, start):
    n = len(nums)
    for d in range(n):
        if nums[start - d] == target:          # start - d có thể âm -> đọc từ cuối!
            return d
        if start + d < n and nums[start + d] == target:
            return d

print("sai :", wrong_no_left_bound([1, 2, 3, 9], 9, 0), "| đúng:", min_distance_expand([1, 2, 3, 9], 9, 0))

sai : 1 | đúng: 3


Với `d = 1`, `start - d = -1` ⇒ `nums[-1] = 9` ⇒ trả 1 thay vì 3. Sửa: `start - d >= 0 and ...`.

### Lỗi 4: chỉ lan một phía

In [26]:
def wrong_right_only(nums, target, start):
    for j in range(start, len(nums)):
        if nums[j] == target:
            return j - start
    return -1

print("sai :", wrong_right_only([5, 0, 0], 5, 2), "| đúng:", min_distance_expand([5, 0, 0], 5, 2))

sai : -1 | đúng: 2


`target` có thể ở bên trái. Sửa: xét cả `start - d`.

## 13. Mẹo nhận dạng pattern

- "Phần tử thỏa điều kiện **gần nhất** với vị trí `p`" ⇒ lan hai phía `p ± d` với `d` tăng dần, dừng ở lần gặp đầu tiên.
- Mảng nhỏ / chỉ hỏi một lần ⇒ quét toàn bộ + `min` là đủ, ít lỗi nhất.
- Nhiều truy vấn trên cùng mảng ⇒ tiền xử lý list chỉ số của mỗi giá trị rồi **binary search** (`bisect`).
- Luôn kiểm biên `0 <= j < n` — chỉ số âm trong Python không báo lỗi.

## 14. Biến thể

1. **Trả về chỉ số gần nhất** (nếu hòa, chọn chỉ số nhỏ hơn).
2. **Nhiều truy vấn** `(target, start)` trên cùng mảng ⇒ dict giá trị → list chỉ số, dùng `bisect`.
3. **LeetCode 821 — Shortest Distance to a Character**: với **mọi** vị trí, khoảng cách tới ký tự `c` gần nhất ⇒ hai lượt quét trái→phải và phải→trái.
4. **Mảng vòng tròn** (LeetCode 2515): khoảng cách `min(|i-s|, n-|i-s|)`.

In [27]:
from bisect import bisect_left
from collections import defaultdict

def build_index(nums):
    pos = defaultdict(list)
    for i, x in enumerate(nums):
        pos[x].append(i)        # chỉ số tự động tăng dần
    return pos

def query(pos, target, start):
    arr = pos[target]
    k = bisect_left(arr, start)          # vị trí chèn start để arr vẫn tăng dần
    best = float("inf")
    if k < len(arr):
        best = min(best, arr[k] - start)
    if k > 0:
        best = min(best, start - arr[k - 1])
    return best

nums = [4, 9, 2, 9, 7, 1, 4]
pos = build_index(nums)
print(dict(pos))
print(query(pos, 4, 3), query(pos, 9, 0), query(pos, 1, 6))
for t in set(nums):
    for s in range(len(nums)):
        assert query(pos, t, s) == min_distance_scan(nums, t, s)
print("truy vấn bisect khớp quét toàn bộ")

{4: [0, 6], 9: [1, 3], 2: [2], 7: [4], 1: [5]}
3 1 1
truy vấn bisect khớp quét toàn bộ


- `defaultdict(list)`: dict tự tạo list rỗng khi gặp key mới.
- `bisect_left(arr, x)`: trong list **đã sắp xếp**, trả vị trí đầu tiên `k` mà `arr[k] >= x` — tìm trong `O(log m)`.
- Phần tử gần `start` nhất chỉ có thể là `arr[k]` (bên phải gần nhất) hoặc `arr[k-1]` (bên trái gần nhất).

## 15. Bài tập (tự làm trước)

**Bài 1 — tính tay:** `nums=[3,8,3,8,3]`, `target=8`, `start=4`. Đáp án? Cách lan hai phía dừng ở `d` nào?

**Bài 2 — dự đoán output:**

```python
nums = [10, 20, 30]
for d in range(3):
    j = 0 - d
    print(j, 0 <= j < len(nums), nums[j])
```

**Bài 3 — điền TODO:** cách lan hai phía.

In [28]:
def exercise_min_distance(nums, target, start):
    # TODO 1: n = len(nums)
    # TODO 2: for d in range(n)
    # TODO 3: kiểm start - d (nhớ >= 0) và start + d (nhớ < n)
    pass

# assert exercise_min_distance([1, 2, 3, 4, 5], 5, 3) == 1
# assert exercise_min_distance([5, 3, 6], 5, 2) == 2

**Bài 4 — biến thể:** trả về **chỉ số** gần `start` nhất; hòa thì lấy chỉ số nhỏ hơn.

In [29]:
def exercise_closest_index(nums, target, start):
    # TODO: lan hai phía, kiểm bên trái TRƯỚC để ưu tiên chỉ số nhỏ
    pass

# assert exercise_closest_index([4, 9, 2, 9, 7, 1, 4], 4, 3) == 0
# assert exercise_closest_index([1, 2, 3, 4, 5], 5, 3) == 4

**Bài 5 — LeetCode 821:** với chuỗi `s` và ký tự `c` (có trong `s`), trả về list `ans` với `ans[i]` = khoảng cách từ `i` tới `c` gần nhất.

In [30]:
def exercise_shortest_to_char(s, c):
    # TODO: lượt trái->phải lưu vị trí c gần nhất bên trái; lượt phải->trái tương tự
    pass

# assert exercise_shortest_to_char("loveleetcode", "e") == [3,2,1,0,1,0,0,1,2,2,1,0]
# assert exercise_shortest_to_char("aaab", "b") == [3, 2, 1, 0]

**Bài 6 — mini challenge (mảng vòng):** mảng chuỗi `words` coi như vòng tròn; từ `startIndex` được đi trái hoặc phải (vòng qua đầu/cuối). Trả về số bước ít nhất tới `target`, hoặc `-1` nếu không có.

In [31]:
def exercise_circular_distance(words, target, startIndex):
    # TODO: với mỗi i có words[i] == target, khoảng cách = min(|i - s|, n - |i - s|)
    pass

# assert exercise_circular_distance(["hello","i","am","leetcode","hello"], "hello", 1) == 1
# assert exercise_circular_distance(["a","b","leetcode"], "leetcode", 0) == 1
# assert exercise_circular_distance(["i","eat","leetcode"], "ate", 0) == -1

## 16. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | 8 ở chỉ số 1 và 3 | `|3-4|=1` | `d=1`, bên trái gặp trước |
| 2 | chỉ số âm đọc từ cuối | `0 <= -1` là False | in `10 20 30` theo thứ tự `0, -1, -2` |
| 3 | `d` tăng dần | hai `if`, mỗi `if` kiểm biên trước | `return d` |
| 4 | như bài 3 | trả `start - d` / `start + d` | trái trước phải |
| 5 | hai lượt quét | `prev = -inf` rồi `i - prev` | lượt ngược lấy `min(ans[i], prev - i)` |
| 6 | khoảng cách vòng | `n - |i - s|` là đi vòng | `min` trên mọi `i`, rỗng thì `-1` |

## 17. Đáp án tham khảo

**Bài 1:** đáp án `1` (chỉ số 3); lan hai phía dừng ở `d = 1`, bên trái.

**Bài 2:** in `0 True 10`, `-1 False 30`, `-2 False 20`.

In [32]:
print(min_distance_expand([3, 8, 3, 8, 3], 8, 4))
nums = [10, 20, 30]
for d in range(3):
    j = 0 - d
    print(j, 0 <= j < len(nums), nums[j])

1
0 True 10
-1 False 30
-2 False 20


In [33]:
def answer_min_distance(nums, target, start):
    n = len(nums)
    for d in range(n):
        if start - d >= 0 and nums[start - d] == target:
            return d
        if start + d < n and nums[start + d] == target:
            return d
    return -1


def answer_closest_index(nums, target, start):
    n = len(nums)
    for d in range(n):
        if start - d >= 0 and nums[start - d] == target:
            return start - d
        if start + d < n and nums[start + d] == target:
            return start + d
    return -1


def answer_shortest_to_char(s, c):
    n = len(s)
    ans = [0] * n
    prev = float("-inf")
    for i in range(n):
        if s[i] == c:
            prev = i
        ans[i] = i - prev
    prev = float("inf")
    for i in range(n - 1, -1, -1):
        if s[i] == c:
            prev = i
        ans[i] = min(ans[i], prev - i)
    return ans


def answer_circular_distance(words, target, startIndex):
    n = len(words)
    best = -1
    for i, w in enumerate(words):
        if w == target:
            d = abs(i - startIndex)
            d = min(d, n - d)
            if best == -1 or d < best:
                best = d
    return best


assert answer_min_distance([1, 2, 3, 4, 5], 5, 3) == 1
assert answer_min_distance([5, 3, 6], 5, 2) == 2
assert answer_closest_index([4, 9, 2, 9, 7, 1, 4], 4, 3) == 0
assert answer_closest_index([1, 2, 3, 4, 5], 5, 3) == 4
assert answer_shortest_to_char("loveleetcode", "e") == [3, 2, 1, 0, 1, 0, 0, 1, 2, 2, 1, 0]
assert answer_shortest_to_char("aaab", "b") == [3, 2, 1, 0]
assert answer_circular_distance(["hello", "i", "am", "leetcode", "hello"], "hello", 1) == 1
assert answer_circular_distance(["a", "b", "leetcode"], "leetcode", 0) == 1
assert answer_circular_distance(["i", "eat", "leetcode"], "ate", 0) == -1
print("Đáp án tham khảo PASS toàn bộ test!")

Đáp án tham khảo PASS toàn bộ test!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def getMinDistance(self, nums: List[int], target: int, start: int) -> int:
        n = len(nums)
        for d in range(n):                                   # khoảng cách tăng dần
            if start - d >= 0 and nums[start - d] == target:
                return d
            if start + d < n and nums[start + d] == target:
                return d
        return -1
```

Phiên bản ngắn: `return min(abs(i - start) for i, x in enumerate(nums) if x == target)`.

Checklist:

- [ ] Trả **khoảng cách**, không phải chỉ số.
- [ ] Có `abs(...)`.
- [ ] Kiểm biên trái `>= 0` **trước** khi đọc (chỉ số âm không báo lỗi!).
- [ ] Xét **cả hai phía**.
- [ ] Đã `from typing import List` khi chạy ngoài LeetCode.